In [0]:
from pyspark.sql.functions import col, date_format

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df_bronze = spark.table("scottish_housing_ws.bronze.google_trends")
df_bronze.printSchema()

In [0]:
df_silver = (
    df_bronze
    .filter(col("is_partial") == False)
    .withColumn("report_date", col("week_start_date").cast("date"))
    .withColumn("year_month", date_format(col("week_start_date"), "yyyy-MM"))
    .drop("week_start_date")
    .select(
        "report_date",
        "year_month",
        "house_prices_scotland",
        "mortgage_edinburgh",
        "rightmove_scotland",
        "espc_edinburgh",
        "property_for_sale_edinburgh",
        "is_partial"
    )
)

In [0]:
df_silver.orderBy("report_date").show(10, truncate=False)

In [0]:
from pyspark.sql.functions import min, max, count
df_silver.agg(
    count("*").alias("row_count"),
    min("report_date").alias("earliest"),
    max("report_date").alias("latest")
).show()

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.silver.google_trends")
)

In [0]:
# Weekly Google Trends index values for 5 Scottish housing market keywords.
# Geography: GB-SCT (Scotland). Timeframe: today 5-y.
# All values are 0-100 relative index, scaled within this single request.
# is_partial rows dropped -- incomplete weeks would skew monthly aggregations in gold.
# year_month added as a join key to monthly silver tables.
# Known limitation: single request normalisation means values are only comparable
# within this keyword set, not against external benchmarks.